##### 초기화

In [5]:
import os

from agent_loop import (
    create_client,
    create_state,
    run_turn,
    print_trace,
)

from tools import (
    reset_memory_store,
    load_memories
)



h:\shs\projects\agent-ref\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
client = create_client()
model = os.getenv("MODEL_NAME") or client.models.list().data[0].id
state = create_state()
reset_memory_store()

##### 도구 능력 밖의 질문:
 - planner가 불가능한 계획(날씨 검색)을 세우는 대신 "무엇이 불가능한지 확인하는" 한 단계 계획으로 끝내야 한다

In [7]:
question = "서울의 오늘 날씨를 알려줘요"
outcome = run_turn(client, model, state, question, debug=False)

print(f"\n[{outcome['status']} / {outcome['llm_calls']} LLM calls / {outcome['replans']} replans]")
print("\n[Final Answer]\n")
print(outcome["content"])
print("\n[Trace]\n")
print_trace(outcome["trace"])

h:\shs\projects\agent-ref\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ssh15\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
h:\shs\projects\agent-ref\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingfa


[completed / 2 LLM calls / 0 replans]

[Final Answer]

죄송합니다만, 현재 내장된 실행 기능 (내부 문서 검색, 사용자 메모 읽기/쓰기, 증거에 대한 추론) 에는 웹 검색이나 날씨 데이터 접근 기능이 포함되어 있지 않아 서울의 오늘 날씨를 확인해 드릴 수 없습니다.

[Trace]

01. {"event": "turn_started", "goal": "서울의 오늘 날씨를 알려줘요"}
02. {"event": "plan_created", "plan": [{"id": 1, "instruction": "Determine that the user's request for today's weather in Seoul cannot be fulfilled because the available Executor capabilities (searching internal company documents, reading/writing user memory, reasoning over evidence) do not include web search, weather data access, or any external tools.", "depends_on": [], "status": "pending"}]}
03. {"event": "plan_step_started", "plan_step_id": 1, "instruction": "Determine that the user's request for today's weather in Seoul cannot be fulfilled because the available Executor capabilities (searching internal company documents, reading/writing user memory, reasoning over evidence) do not include web search, weather data access, or any external tools.

In [8]:
memories = load_memories()
for memory in memories:
    print(f"\n[Memory]\n{memory['content']}\n")


[Memory]
The user's request for today's weather in Seoul cannot be fulfilled because the available Executor capabilities (searching internal company documents, reading/writing user memory, reasoning over evidence) do not include web search, weather data access, or any external tools.



##### 정상 과제:
 - 다단계 계획 → 문서 검색 → 근거 있는 최종 답변


In [9]:
question = (
    "2026년 해외 출장의 숙박비와 식비 한도, 그리고 출발 전 승인 요건을 "
    "근거 문서와 함께 정리해줘."
)

question = (
    "2026년 해외 출장의 숙박비와 식비 한도, 그리고 출발 전 승인 요건을 정리해줘."
)
outcome = run_turn(client, model, state, question, debug=False)

print(f"\n[{outcome['status']} / {outcome['llm_calls']} LLM calls / {outcome['replans']} replans]")
print("\n[Final Answer]\n")
print(outcome["content"])
print("\n[Trace]\n")
print_trace(outcome["trace"])


[completed / 6 LLM calls / 0 replans]

[Final Answer]

# 2026 년 해외 출장 정책 보고서

## 1. 비용 한도 기준

### 숙박비
- **한도**:每晚 최대 220 달러
- **적용 범위**: 해외 비즈니스 출장 시
- **출처**: `travel_lodging_B.txt` (2026 년, active)

### 식비
- **한도**: 일일 최대 60 달러
- **적용 범위**: 승인된 비즈니스 여행 중
- **출처**: `meal_expense.txt` (2026 년, active)

## 2. 출발 전 승인 요건

### 사전 승인 절차
- **필수 조건**: 국제 비즈니스 출장 요청은 출발 전에 부서장 (department head) 의 승인을 받아야 합니다.
- **출처**: `travel_request.txt` (2026 년, active)

### 필요한 서류
- 제공된 내부 문서에는 구체적인 서류 목록이 명시되지 않았습니다. 추가적인 세부 사항이 필요한 경우 관련 문서를 더 자세히 확인하거나 추가 정보를 요청할 수 있습니다.

## 3. 참고 사항
- 모든 기준은 2026 년 기준 정책입니다.
- 위 정보는 `travel_lodging_B.txt`, `meal_expense.txt`, `travel_request.txt` 문서에서 확인된 내용입니다.

[Trace]

01. {"event": "turn_started", "goal": "2026년 해외 출장의 숙박비와 식비 한도, 그리고 출발 전 승인 요건을 정리해줘."}
02. {"event": "plan_created", "plan": [{"id": 1, "instruction": "검색: 2026 년 해외 출장 관련 정책 문서에서 숙박비와 식비 한도 기준을 찾아라.", "depends_on": [], "status": "pending"}, {"id": 2, "instruction": "검색: 2026 년 해외 출장 관련 정책 문서에서 출발 